# Stage 2 - Claim extraction and hallucination detection
Pipeline: **answer -> atomic claims -> detector score per claim -> answer-level decision**. Every score is in [0,1] (higher = more likely hallucinated).

| signal | definition | needs |
|---|---|---|
| `s_nli` | 1 - max entailment of the claim given retrieved evidence (DeBERTa-v3 NLI cross-encoder) | retrieval |
| `s_llm` | LLM verifier verdict: SUPPORTED 0 / NOT_ENOUGH_INFO 0.5 / CONTRADICTED 1 | retrieval |
| `s_sc` | SelfCheckGPT-NLI: mean over N samples (T=0.7) of P(contra)/(P(contra)+P(entail)) | N extra samples |
| `s_cove` | Chain-of-Verification without retrieval: independent verification questions | 3 LLM calls |

Detectors: `nli`, `llm`, `selfcheck`, `cove`, `evidence` (nli+llm), `hybrid` (nli+llm+selfcheck; logistic regression). Thresholds are fitted on the HaluEval **dev** split with **Youden's J** (prevalence-invariant - F1-optimal thresholds collapse to "flag everything" for weak detectors on a 50/50 benchmark) and reported on the disjoint **test** split. RAGTruth and FEVER are zero-shot transfer.

In [ ]:
import os, sys
from pathlib import Path
import matplotlib; matplotlib.use("Agg")          # figures are written to results/ and shown with IPython.display.Image
from IPython.display import Image, display, Markdown
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", "{:.3f}".format)

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# ---- choose how to run -----------------------------------------------------------------------------------------
PROFILE = None          # None = configs/default.yaml | "configs/quick.yaml" = small sanity run (recommended first)
MOCK = os.environ.get("HALLUGUARD_MOCK") == "1"      # offline smoke test with FAKE models - numbers are meaningless
from halluguard.config import load_config
cfg = load_config(PROFILE, mock=MOCK)
print("mock:", MOCK, "| generator:", cfg.llm.model, "| judge:", cfg.llm.judge_model, "| results ->", cfg.project.results_dir)

def show(name, stage):
    p = Path(cfg.project.results_dir) / stage / name
    display(Image(filename=str(p))) if p.exists() else print("(missing)", p)


In [ ]:
from halluguard.budget import print_budget
print_budget(cfg)            # rough token budget BEFORE you start (runs are resumable: finished LLM calls are cached)

## 1. Claim extraction, one example
A bare entity answer ("Arthur's Magazine") is meaningless without its question, so claims are *decontextualised*.

In [ ]:
from halluguard.experiments.common import load_resources
from halluguard.pipeline import FactualityPipeline, QuestionState
res = load_resources(cfg)
pipe = FactualityPipeline(cfg, res.llm, res.retriever, res.nli)
it = res.splits["halueval_dev"][0]
st = QuestionState(it["id"], it["question"]); st.answers = {"R": it["right_answer"], "H": it["hallucinated_answer"]}
for which in ("R", "H"):
    rows = pipe.features(st, which, ["s_nli", "s_llm", "s_sc"])
    print(f"[{which}] answer = {st.answers[which]!r}")
    for r in rows:
        print("   claim:", r["claim"]); print("   scores:", {k: round(v, 3) for k, v in r["feats"].items() if v is not None})
        print("   top evidence:", (r["evidence"][0]["text"][:120] + "...") if r["evidence"] else "-")

## 2. Run the full Stage 2 benchmark
HaluEval (answer level, dev/test), RAGTruth QA (response level), FEVER (3-way). Writes `results/stage2/` incl. `calibration.json`, which Stage 3 loads.

In [ ]:
from halluguard.experiments import stage2
r2 = stage2.run(cfg, res)
cols = ["dataset","detector","n","pos_rate","threshold","accuracy","precision","recall","f1","auroc","auprc"]
r2["metrics"][cols]

### Detector comparison and ROC / PR (HaluEval test)

In [ ]:
show("detector_comparison.png", "stage2"); show("roc_pr_halueval.png", "stage2")

### Does retrieval quality limit detection? Gold paragraph vs. bm25 / dense / hybrid evidence

In [ ]:
import pandas as pd
d = pd.read_csv(Path(cfg.project.results_dir) / "stage2" / "retrieval_mode_vs_detection.csv")
d[["evidence","auroc","auprc","f1","precision","recall"]]

### How many samples does SelfCheck need?

In [ ]:
show("selfcheck_n_curve.png", "stage2")

### RAGTruth (naturally generated RAG hallucinations) and FEVER (3-way verification)

In [ ]:
rt = r2["ragtruth"]; display(rt["metrics"] if rt else "RAGTruth skipped")
show("roc_pr_ragtruth.png", "stage2"); show("fever_confusion.png", "stage2")

### Confusion matrices and score distributions (HaluEval test)

In [ ]:
show("confusion_halueval.png", "stage2"); show("score_hist_halueval.png", "stage2")

### Error analysis
`results/stage2/error_analysis.md` lists the most confident false positives / false negatives of the best detector together with the retrieved evidence and the gold paragraph.

In [ ]:
print((Path(cfg.project.results_dir) / "stage2" / "error_analysis.md").read_text()[:3500])

### Calibrated detector heads (used by Stage 3)

In [ ]:
pd.DataFrame([h.to_dict() for h in r2["calibration"].heads.values()])[["name","features","kind","weights","bias","threshold"]]

## Reading the results
* Compare `evidence` / `hybrid` against `nli`, `llm`, `selfcheck`, `cove` - *which signal carries the detection, and does fusion help?* (look at AUROC/AUPRC, which do not depend on thresholds).
* `nli_gold` vs. `nli` shows how much of the remaining error is due to retrieval rather than verification.
* RAGTruth is imbalanced (see `pos_rate`): prefer AUPRC over accuracy.
* Caveat: thresholds are fitted on answer-level HaluEval labels and applied to individual claims in Stage 3 (HaluEval answers are mostly single-claim, so the scales match).